In [1]:
import os
import math
import pickle
from collections import Counter
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ================================
# 1) CLEAN SPEED & API SETTINGS
# ================================
torch.set_float32_matmul_precision('high')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================================
# 2) VOCAB & DATA PREPARATION
# ================================
PAD, SOS, EOS, UNK = "<PAD>", "<SOS>", "<EOS>", "<UNK>"
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3
special_tokens = [PAD, SOS, EOS, UNK]

def build_vocab(texts):
    counter = Counter()
    for t in texts: counter.update(list(t))
    vocab = {tok: i for i, tok in enumerate(special_tokens)}
    for ch, _ in counter.most_common():
        if ch not in vocab: vocab[ch] = len(vocab)
    return vocab

file_path = "20260101_K-M_YiTailvarTolin_Tolgai uguud.txt"
data = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().rsplit(",", 1)
        if len(parts) == 2:
            data.append({"input": parts[0].strip(), "target": parts[1].strip()})
df = pd.DataFrame(data)

# Split 90% Train / 10% Val
split_idx = int(0.9 * len(df))
train_df = df.iloc[:split_idx].reset_index(drop=True)
val_df = df.iloc[split_idx:].reset_index(drop=True)

cyr_vocab = build_vocab(df["input"].tolist())
mng_vocab = build_vocab(df["target"].tolist())
inv_mng_vocab = {v: k for k, v in mng_vocab.items()}

# ================================
# 3) DATASET & MODEL
# ================================
class TranslationDataset(Dataset):
    def __init__(self, df, src_vocab, tgt_vocab):
        self.src = df["input"].tolist()
        self.tgt = df["target"].tolist()
        self.src_vocab, self.tgt_vocab = src_vocab, tgt_vocab
    def __len__(self): return len(self.src)
    def __getitem__(self, idx):
        s = [SOS_IDX] + [self.src_vocab.get(c, UNK_IDX) for c in self.src[idx]] + [EOS_IDX]
        t = [SOS_IDX] + [self.tgt_vocab.get(c, UNK_IDX) for c in self.tgt[idx]] + [EOS_IDX]
        return torch.tensor(s), torch.tensor(t)

def collate_fn(batch):
    src_list, tgt_list = zip(*batch)
    src_pad = nn.utils.rnn.pad_sequence(src_list, batch_first=True, padding_value=PAD_IDX)
    tgt_pad = nn.utils.rnn.pad_sequence(tgt_list, batch_first=True, padding_value=PAD_IDX)
    return src_pad, tgt_pad

train_loader = DataLoader(TranslationDataset(train_df, cyr_vocab, mng_vocab), batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(TranslationDataset(val_df, cyr_vocab, mng_vocab), batch_size=64, shuffle=False, collate_fn=collate_fn)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class TransformerSeq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, nhead=8, num_layers=4):
        super().__init__()
        self.d_model = d_model
        self.src_emb = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD_IDX)
        self.tgt_emb = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_enc = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, 
            num_encoder_layers=num_layers, num_decoder_layers=num_layers,
            dim_feedforward=1024, dropout=0.1, batch_first=True
        )
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt):
        tgt_mask = self.transformer.generate_square_subsequent_mask(tgt.size(1)).to(device)
        # Ensure masks are boolean to avoid UserWarnings
        src_key_mask = (src == PAD_IDX)
        tgt_key_mask = (tgt == PAD_IDX)
        
        src_emb = self.pos_enc(self.src_emb(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_enc(self.tgt_emb(tgt) * math.sqrt(self.d_model))
        
        return self.fc_out(self.transformer(
            src_emb, tgt_emb, tgt_mask=tgt_mask, 
            src_key_padding_mask=src_key_mask, 
            tgt_key_padding_mask=tgt_key_mask,
            memory_key_padding_mask=src_key_mask
        ))

# ================================
# 4) OPTIMIZATION & METRICS
# ================================
model = TransformerSeq2Seq(len(cyr_vocab), len(mng_vocab)).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=2e-4, steps_per_epoch=len(train_loader), epochs=50, pct_start=0.1         # Epoch 50
)
scaler = torch.amp.GradScaler('cuda')

def get_cer(model, loader):
    model.eval()
    errors = 0
    total_chars = 0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            output = model(src, tgt[:, :-1])
            preds = output.argmax(dim=-1)
            for i in range(preds.size(0)):
                p = "".join([inv_mng_vocab.get(idx.item(), "") for idx in preds[i] if idx not in special_tokens])
                t = "".join([inv_mng_vocab.get(idx.item(), "") for idx in tgt[i] if idx not in special_tokens])
                errors += len(set(p) ^ set(t)) # Simple char diff for speed
                total_chars += max(len(t), 1)
    return errors / total_chars

# ================================
# 5) TRAINING LOOP
# ================================


best_val_loss = float('inf')
epochs_no_improve = 0

for epoch in range(1, 51):                  # Epoch 50
    model.train()
    total_train_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    
    for src, tgt in pbar:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            output = model(src, tgt[:, :-1])
            loss = criterion(output.reshape(-1, output.size(-1)), tgt[:, 1:].reshape(-1))
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_train_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    # Validation Phase
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device), tgt.to(device)
            with torch.amp.autocast('cuda'):
                output = model(src, tgt[:, :-1])
                loss = criterion(output.reshape(-1, output.size(-1)), tgt[:, 1:].reshape(-1))
                val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    print(f"✅ Epoch {epoch} Summary: Train Loss: {total_train_loss/len(train_loader):.4f} | Val Loss: {avg_val_loss:.4f}")

    # Early Stopping & Saving Best
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'cyr_vocab': cyr_vocab, 'mng_vocab': mng_vocab,
            'config': {'d_model': 256, 'nhead': 8, 'num_encoder_layers': 4, 'dim_feedforward': 1024}
        }, "cyr2mng_best_model_Gemini3.pth")
        print("⭐ New Best Model Saved!")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= 5:
            print("Early stopping triggered.")
            break

print("Training Complete.")

C:\Users\tsogn\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:85.)
  _C._set_float32_matmul_precision(precision)
Epoch 1:   0%|                                                                                    | 0/1839 [00:00<?, ?it/s]C:\Users\tsogn\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:6044: UserWarning: Support for mismatched key_padding_mask and 

✅ Epoch 1 Summary: Train Loss: 2.4527 | Val Loss: 2.0419
⭐ New Best Model Saved!


Epoch 2: 100%|████████████████████████████████████████████████████████████| 1839/1839 [01:44<00:00, 17.63it/s, loss=1.7208]


✅ Epoch 2 Summary: Train Loss: 1.8864 | Val Loss: 1.7997
⭐ New Best Model Saved!


Epoch 3: 100%|████████████████████████████████████████████████████████████| 1839/1839 [01:46<00:00, 17.20it/s, loss=1.5710]


...

✅ Epoch 20 Summary: Train Loss: 0.8957 | Val Loss: 0.9797


Epoch 21: 100%|███████████████████████████████████████████████████████████| 1839/1839 [01:40<00:00, 18.21it/s, loss=0.9256]


✅ Epoch 21 Summary: Train Loss: 0.8957 | Val Loss: 0.9797


Epoch 22: 100%|██████████████████████████████████████████████████████████████| 1839/1839 [01:39<00:00, 18.43it/s, loss=nan]


✅ Epoch 22 Summary: Train Loss: nan | Val Loss: nan
Early stopping triggered.
Training Complete.


In [ ]:
import torch
import torch.nn as nn
import math
import os

# ================================
# 1) ARCHITECTURE (Must match Training)
# ================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerSeq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, nhead=8, num_layers=4, dim_feedforward=1024):
        super().__init__()
        self.d_model = d_model
        self.src_emb = nn.Embedding(src_vocab_size, d_model, padding_idx=0)
        self.tgt_emb = nn.Embedding(tgt_vocab_size, d_model, padding_idx=0)
        self.pos_enc = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, 
            num_encoder_layers=num_layers, num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward, batch_first=True
        )
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt):
        tgt_mask = self.transformer.generate_square_subsequent_mask(tgt.size(1)).to(src.device)
        src_emb = self.pos_enc(self.src_emb(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_enc(self.tgt_emb(tgt) * math.sqrt(self.d_model))
        out = self.transformer(src_emb, tgt_emb, tgt_mask=tgt_mask, 
                               src_key_padding_mask=(src == 0), 
                               tgt_key_padding_mask=(tgt == 0),
                               memory_key_padding_mask=(src == 0))
        return self.fc_out(out)

# ================================
# 2) LOAD MODEL & VOCAB
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = "cyr2mng_best_model_Gemini3.pth"

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"Checkpoint not found at {checkpoint_path}. Please check the filename.")

checkpoint = torch.load(checkpoint_path, map_location=device)
cyr_vocab = checkpoint['cyr_vocab']
mng_vocab = checkpoint['mng_vocab']
config = checkpoint['config']
inv_mng_vocab = {v: k for k, v in mng_vocab.items()}

# Special Token Indices
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3
special_tokens = [PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX]

model = TransformerSeq2Seq(
    src_vocab_size=len(cyr_vocab),
    tgt_vocab_size=len(mng_vocab),
    d_model=config['d_model'],
    nhead=config['nhead'],
    num_layers=config['num_encoder_layers'],
    dim_feedforward=config['dim_feedforward']
).to(device)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"✅ Best model loaded from {checkpoint_path}")

# ================================
# 3) BEAM SEARCH WITH LENGTH PENALTY
# ================================


def translate_beam(src_text, beam_size=5, alpha=0.7, max_len=100):
    model.eval()
    with torch.no_grad():
        # Encode input characters
        tokens = [SOS_IDX] + [cyr_vocab.get(ch, UNK_IDX) for ch in src_text] + [EOS_IDX]
        src = torch.tensor(tokens).unsqueeze(0).to(device)
        
        # beams list: [(sequence, cumulative_log_prob)]
        beams = [([SOS_IDX], 0.0)]
        
        for _ in range(max_len):
            candidates = []
            for seq, score in beams:
                # If a beam reaches EOS, keep it as a candidate
                if seq[-1] == EOS_IDX:
                    candidates.append((seq, score))
                    continue
                
                tgt_in = torch.tensor(seq).unsqueeze(0).to(device)
                logits = model(src, tgt_in)
                log_probs = torch.log_softmax(logits[0, -1, :], dim=-1)
                
                # Expand top k next tokens
                top_v, top_i = torch.topk(log_probs, beam_size)
                for i in range(beam_size):
                    candidates.append((seq + [top_i[i].item()], score + top_v[i].item()))
            
            # Ranking with Length Penalty: (Score / (Length^alpha))
            # This prevents the model from cutting off long Mongolian suffixes too early.
            def lp_score(cand):
                s, sc = cand
                penalty = ((5 + len(s))**alpha) / ((5 + 1)**alpha)
                return sc / penalty

            beams = sorted(candidates, key=lp_score, reverse=True)[:beam_size]
            
            # If all top beams hit EOS, we stop
            if all(b[0][-1] == EOS_IDX for b in beams):
                break

    best_seq = beams[0][0]
    # Filter out special tokens and build string
    return "".join([inv_mng_vocab.get(idx, "") for idx in best_seq if idx not in special_tokens])

# ================================
# 4) INTERACTIVE INTERFACE
# ================================
print("\n" + "="*40)
print("  CYRILLIC -> MONGOLIAN SCRIPT")
print("="*40)
print("Enter a word to translate. Type 'q' to quit.")

while True:
    txt = input("\nCyrillic: ").strip()
    if txt.lower() == 'q':
        break
    if not txt:
        continue
    
    # Translate
    result = translate_beam(txt, beam_size=5, alpha=0.7)
    print(f"Mongolian: {result}")

✅ Best model loaded from cyr2mng_best_model_Gemini3.pth

  CYRILLIC -> MONGOLIAN SCRIPT
Enter a word to translate. Type 'q' to quit.



Cyrillic:  МЭДЭЭ


C:\Users\tsogn\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(
C:\Users\tsogn\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
C:\Users\tsogn\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both inste

Mongolian: ᠮᠡᠳᠡᠭᠡ



Cyrillic:  МЭДЭЭ


...

Mongolian: ᠪᠢᠴᠢᠬᠦ ᠳᠦᠷᠢᠮ ᠦᠨ



Cyrillic:  ЖУРАМЛАСАН ТОЛЬ


Mongolian: ᠵᠢᠷᠤᠮᠯᠠᠭᠰᠠᠨ ᠣᠯᠢᠲᠠ



Cyrillic:  ЖУРАМЛАСАН ТОЛЬ


Mongolian: ᠵᠢᠷᠤᠮᠯᠠᠭᠰᠠᠨ ᠣᠯᠢᠲᠠ
